In [ ]:
from google import genai
from google.genai.types import HttpOptions

import os
from dotenv import load_dotenv
from google import genai

load_dotenv()  # loads .env into environment

api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)

response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=[
        {
            "role": "user",
            "parts": [{"text": "Explain 2+2"}]
        }
    ]
)

print(response)

In [ ]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=420,
  prompt_token_count=6,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=6
    ),
  ],
  thoughts_token_count=878,
  total_token_count=1304
)

'4'

In [1]:
from models.google_api import GoogleModel
from multi_agent.multi_agent import Problem, Role

import os
from pathlib import Path
from models.google_api import GoogleClient, google_assistant_format, google_user_format
model_name = "gemini-2.5-flash"





gemini_25_flash = GoogleClient().select_model(model_name)
messages = google_user_format("What is 2+2")


def conversation_google(model:GoogleModel, name:str, n_steps: int, problem: Problem, wait:int = 5, continue_from: str | None = None):
      if continue_from is None:
        problem.reset_cycle()
        roles = problem.all_roles
        raws = []
        messages = [google_user_format(problem.pose_problem())]

      else:
        messages, raws = GoogleModel.load_conv(continue_from)  

      for step in range(n_steps):
        print(f"\nSTEP {step}: \n")
        next_agent = problem.next_agent()
        user_msg = f"\nWhat do you say, {next_agent.name}?\n"
        reply, raw = model.send_msg_and_get_contnent(messages+[google_user_format(user_msg)], next_agent.instruction())
        raws.append(raw)
        print(f"\nAgent: {next_agent}\n {reply}")
        messages.append(google_assistant_format(reply))
        path = GoogleModel.save_conv(name=name, raw=raws, messages=messages)
      print(f"\n{model.compute_token_cost()}")
      return messages, raws, path

Matematician = Role(
    name="Concise solver",
    behavior="You solve problems concisely"
  )
Verifier = Role(
    name="Verifier",
    behavior="You repeat what the matematician said. Then you say LGTM"
  )

twoplustwo=Problem(
  roles = [Matematician, Verifier],
  problem_descr="What is 2+2?",
  answer=4
)
print(twoplustwo)

messages = conversation_google(model = gemini_25_flash, name="gemini_25_flash_test_run",problem= twoplustwo, n_steps=2)


Welcome Concise solver, and Verifier. Together, you should solve the following
problem: >> What is 2+2?.<<  "When you are done, you should submidt your answer
as: ANSWER: <your answer>.  No latex formatting, just the raw number/numbers or
strings at the very end.  Before you start sharing your toughts, give a little
summary of the conversation so far.  Give a list of the currently suggested
answers. Everytime you propose an aswer, check this list.  You proposal cannot
be in this this list. Try again and submit a new unique answer."

STEP 0: 


Agent: Role: Concise solver
You solve problems concisely
 Summary of conversation so far:
The user welcomed the Concise solver and Verifier, then asked "What is 2+2?". They provided specific instructions on how to format the final answer (ANSWER: <your answer>, no latex), and how to manage proposed answers (maintain a list, new proposals must be unique).

Currently suggested answers:
[]

I say that 2+2 is 4. My proposal is not in the list of curr

In [16]:
x, y, z = messages


In [17]:
x

[{'role': 'user',
  'parts': [{'text': 'Welcome Concise solver, and Verifier.\nTogether, you should solve the following problem: >>\nWhat is 2+2?.<<\n\n"When you are done, you should submidt your answer as: ANSWER: <your answer>. \nNo latex formatting, just the raw number/numbers or strings at the very end. \nBefore you start sharing your toughts, give a little summary of the conversation so far. \nGive a list of the currently suggested answers. Everytime you propose an aswer, check this list. \nYou proposal cannot be in this this list. Try again and submit a new unique answer."\n\n'}]},
 {'role': 'model',
  'parts': [{'text': 'Summary of the conversation so far: The user has initiated the problem-solving process by asking "What is 2+2?" and has provided specific instructions for the answer format and collaboration with a "Verifier".\n\nCurrently suggested answers: None.\n\nMy proposal: The sum of 2 and 2 is 4.\n\nANSWER: 4'}]},
 {'role': 'model',
  'parts': [{'text': 'The mathematicia

In [21]:
y

GenerateContentResponse(
  automatic_function_calling_history=[],
  candidates=[
    Candidate(
      content=Content(
        parts=[
          Part(
            text="""The mathematician said that the sum of 2 and 2 is 4.
LGTM"""
          ),
        ],
        role='model'
      ),
      finish_reason=<FinishReason.STOP: 'STOP'>,
      index=0
    ),
  ],
  model_version='gemini-2.5-flash',
  response_id='Ykkxab2mFdqXvdIP4PrRwAU',
  sdk_http_response=HttpResponse(
    headers=<dict len=11>
  ),
  usage_metadata=GenerateContentResponseUsageMetadata(
    candidates_token_count=19,
    prompt_token_count=237,
    prompt_tokens_details=[
      ModalityTokenCount(
        modality=<MediaModality.TEXT: 'TEXT'>,
        token_count=237
      ),
    ],
    thoughts_token_count=101,
    total_token_count=357
  )
)

In [22]:
gemini_25_flash.num_tokens

[GenerateContentResponseUsageMetadata(
   candidates_token_count=70,
   prompt_token_count=156,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=156
     ),
   ],
   thoughts_token_count=174,
   total_token_count=400
 ),
 GenerateContentResponseUsageMetadata(
   candidates_token_count=19,
   prompt_token_count=237,
   prompt_tokens_details=[
     ModalityTokenCount(
       modality=<MediaModality.TEXT: 'TEXT'>,
       token_count=237
     ),
   ],
   thoughts_token_count=101,
   total_token_count=357
 )]

In [24]:
from collections import defaultdict

token_counts = defaultdict(int)   


for u in gemini_25_flash.num_tokens:
    print(
        "prompt:", u.prompt_token_count,
        "candidates:", u.candidates_token_count,
        "thoughts:", u.thoughts_token_count,
        "total:", u.total_token_count,
    )

for u in gemini_25_flash.num_tokens:
    
        token_counts["prompt"] += u.prompt_token_count
        token_counts["candidates"] += u.candidates_token_count
        token_counts["thoughts"] += u.thoughts_token_count
        token_counts["total"] += u.total_token_count
    

print(token_counts)



prompt: 156 candidates: 70 thoughts: 174 total: 400
prompt: 237 candidates: 19 thoughts: 101 total: 357
defaultdict(<class 'int'>, {'prompt': 393, 'candidates': 89, 'thoughts': 275, 'total': 757})
